# Notebook 3: Hugging Face Advanced Use Cases for Workshop

This notebook focuses on advanced generation activities in a course format:

- Essay generation
- Story writing with style control
- Medical report drafting with MedGemma

All selected models remain <= 8B parameters.

### Learning Goals

1. Build prompt-driven writing workflows with multiple model sizes.
2. Observe practical quality/runtime tradeoffs without formal benchmarks.
3. Apply safe prompting patterns for medical-report drafting demos.

### Session Flow

1. Part A: Essay generation (TinyLlama 1.1B -> Gemma 2B optional -> Qwen 7B optional)
2. Part B: Story generation with style constraints
3. Part C: MedGemma 4B medical report drafting demo

In [ ]:
%pip install -q -U transformers accelerate torch pillow requests huggingface_hub

In [ ]:
import time
import torch
import requests
from PIL import Image
from transformers import pipeline

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Optional for gated models (Gemma / MedGemma):
# from huggingface_hub import notebook_login
# notebook_login()

## Part A: Essay Generation With Increasing Model Sizes

We use the same essay task across progressively larger models.

Class focus:
- Structure and coherence of the essay.
- Prompt-following quality.
- Practical speed/memory differences.

### Step A1: Essay With TinyLlama 1.1B

Model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

This is a lightweight model suitable for local workshop demos.

In [ ]:
essay_prompt = (
    'You are an academic writing assistant. Write a concise essay (350-450 words) on the topic: '
    '"AI literacy should be mandatory in high school." '
    'Use this structure: Introduction, Two Supporting Arguments, Counterargument, Conclusion. '
    'Use clear formal language.'
)

tinyllama_model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

start = time.time()
if torch.cuda.is_available():
    tinyllama_writer = pipeline(
        'text-generation',
        model=tinyllama_model_id,
        device_map='auto',
        torch_dtype=torch.float16
    )
else:
    tinyllama_writer = pipeline('text-generation', model=tinyllama_model_id)

print('Loaded model:', tinyllama_model_id)
print(f'Load time: {time.time() - start:.2f} sec')

tinyllama_essay = tinyllama_writer(
    essay_prompt,
    max_new_tokens=520,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)[0]['generated_text']

if tinyllama_essay.startswith(essay_prompt):
    tinyllama_essay = tinyllama_essay[len(essay_prompt):].strip()

print('Essay output (TinyLlama 1.1B):')
print(tinyllama_essay)

### Step A2 (Optional): Essay With Gemma 2B

Model: `google/gemma-2-2b-it`

This is an intermediate-size model between 1.1B and 7B.

Note: You must accept Gemma terms on Hugging Face first.

In [ ]:
gemma_model_id = 'google/gemma-2-2b-it'
gemma_writer = None

try:
    start = time.time()
    if torch.cuda.is_available():
        gemma_writer = pipeline(
            'text-generation',
            model=gemma_model_id,
            device_map='auto',
            torch_dtype=torch.float16
        )
    else:
        gemma_writer = pipeline('text-generation', model=gemma_model_id)

    print('Loaded model:', gemma_model_id)
    print(f'Load time: {time.time() - start:.2f} sec')

    gemma_essay = gemma_writer(
        essay_prompt,
        max_new_tokens=520,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
)[0]['generated_text']

    if gemma_essay.startswith(essay_prompt):
        gemma_essay = gemma_essay[len(essay_prompt):].strip()

    print('Essay output (Gemma 2B):')
    print(gemma_essay)
except Exception as exc:
    print('Gemma step skipped (likely gated access or resource limits).')
    print('Details:', exc)

### Step A3 (Optional): Essay With Qwen 7B

Model: `Qwen/Qwen2.5-7B-Instruct`

This step demonstrates a larger 7B instruct model for richer writing style.

If resources are limited, skip this step during live delivery.

In [ ]:
qwen_model_id = 'Qwen/Qwen2.5-7B-Instruct'
qwen_writer = None

try:
    start = time.time()
    if torch.cuda.is_available():
        qwen_writer = pipeline(
            'text-generation',
            model=qwen_model_id,
            device_map='auto',
            torch_dtype=torch.float16
        )
    else:
        qwen_writer = pipeline('text-generation', model=qwen_model_id)

    print('Loaded model:', qwen_model_id)
    print(f'Load time: {time.time() - start:.2f} sec')

    qwen_essay = qwen_writer(
        essay_prompt,
        max_new_tokens=520,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
)[0]['generated_text']

    if qwen_essay.startswith(essay_prompt):
        qwen_essay = qwen_essay[len(essay_prompt):].strip()

    print('Essay output (Qwen 7B):')
    print(qwen_essay)
except Exception as exc:
    print('Qwen step skipped due to resource limits.')
    print('Details:', exc)

### Essay Reflection

Ask students:
- Which model followed the structure best?
- Which model gave the best tradeoff for local usage?

Next, we keep the same models for a creative story task.

## Part B: Story Writing With Style Constraints

We now test creativity and instruction-following using a controlled story prompt.

### Step B1: Story With TinyLlama 1.1B

Prompt design idea: explicitly request tone, characters, and ending style.

In [ ]:
story_prompt = (
    'Write a short story (450-600 words) in a hopeful sci-fi style. '
    'Requirements: include a teacher, a low-cost robot, and one moral dilemma. '
    'End with a reflective final paragraph.'
)

tinyllama_story = tinyllama_writer(
    story_prompt,
    max_new_tokens=700,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)[0]['generated_text']

if tinyllama_story.startswith(story_prompt):
    tinyllama_story = tinyllama_story[len(story_prompt):].strip()

print('Story output (TinyLlama 1.1B):')
print(tinyllama_story)

### Step B2 (Optional): Story With Qwen 7B

Run this cell only if Qwen was successfully loaded above.

In [ ]:
if qwen_writer is not None:
    qwen_story = qwen_writer(
        story_prompt,
        max_new_tokens=700,
        do_sample=True,
        temperature=0.8,
        top_p=0.9
)[0]['generated_text']

    if qwen_story.startswith(story_prompt):
        qwen_story = qwen_story[len(story_prompt):].strip()

    print('Story output (Qwen 7B):')
    print(qwen_story)
else:
    print('Qwen story step skipped because model is not loaded.')

## Part C: MedGemma 4B For Medical Report Drafting

Important workshop safety note:

- This is an educational demo only.
- Model output is not medical advice.
- Any real clinical use needs expert review and formal validation.

In [ ]:
medical_image_url = 'https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png'
medical_image = Image.open(requests.get(medical_image_url, stream=True, timeout=30).raw).convert('RGB')
display(medical_image)

### Step C1: Load MedGemma 4B

Model: `google/medgemma-4b-it`

You must accept model terms and authenticate before this step can run.

In [ ]:
medgemma_model_id = 'google/medgemma-4b-it'
medgemma_pipe = None

try:
    start = time.time()
    if torch.cuda.is_available():
        medgemma_pipe = pipeline(
            'image-text-to-text',
            model=medgemma_model_id,
            device_map='auto',
            torch_dtype=torch.bfloat16
        )
    else:
        medgemma_pipe = pipeline('image-text-to-text', model=medgemma_model_id)

    print('Loaded model:', medgemma_model_id)
    print(f'Load time: {time.time() - start:.2f} sec')
except Exception as exc:
    print('MedGemma load skipped (likely gated access or memory constraints).')
    print('Details:', exc)

### Step C2: Generate A Draft Report

Highlighted generation parameters:
- `max_new_tokens`: limits report length.
- `do_sample=False`: more deterministic output for classroom consistency.

In [ ]:
if medgemma_pipe is not None:
    med_messages = [
        {
            'role': 'system',
            'content': [
                {'type': 'text', 'text': 'You are a radiology assistant. Be concise and explicit about uncertainty.'}
            ]
        },
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': medical_image},
                {
                    'type': 'text',
                    'text': 'Generate a draft report with sections: Findings, Impression, and Follow-up Considerations.'
                }
            ]
        }
    ]

    med_result = medgemma_pipe(
        text=med_messages,
        max_new_tokens=260,
        do_sample=False
    )

    print('MedGemma draft report:')
    print(med_result[0]['generated_text'][-1]['content'])
else:
    print('MedGemma generation skipped because model is not loaded.')

## End Of Notebook 3: Suggested Class Activities

1. Ask each student group to choose one model size for essay writing and justify it.
2. Reuse the same story prompt with one changed constraint and compare outputs.
3. Discuss why medical-model outputs must be treated as draft assistance only.